# Preprocess all the matches


#### Libraries

In [267]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

#### Functions

In [268]:
def load_all_df(folder_path) -> pd.DataFrame:
    folder = Path(folder_path)

    df = pd.DataFrame()
    
    for subdir, dirs, files in os.walk(folder):
        for file in tqdm(sorted(files[:-1]), total=len(files)-1):
            if file.endswith('.csv'):
                file_path = os.path.join(subdir, file)
                data = pd.read_csv(file_path)
                df = pd.concat([df, data], axis = 0)

    return df


In [269]:
def clean_df(data):

    tourney_id = data['tourney_id']\
        .astype(str)\
        .rename('tourney_id')
    
    match_year = data['tourney_id']\
        .astype(str)\
        .str[:4]\
        .rename('match_year')
    
    tourney_name = data['tourney_name']\
        .astype(str)\
        .rename('tourney_name')
    
    surface = data['surface']\
        .astype(str)\
        .fillna('Unknown')\
        .replace('nan', 'Unknown')\
        .rename('surface')
    
    draw_size = data['draw_size']\
        .astype(int)\
        .rename('draw_size')
    
    tourney_level = data["tourney_level"]\
        .astype(str)\
        .rename('tourney_level')
    
    tourney_date = data["tourney_date"]\
        .astype(int)\
        .rename('tourney_date')
    
    match_num = data["match_num"]\
        .astype(int)\
        .rename('match_num')
    
    match_id = (data['tourney_id'].astype(str) + data['match_num'].astype(str))\
        .astype(str)\
        .rename('match_id')

    score = data["score"]\
        .astype(str)\
        .replace('nan', 'Unknown')\
        .rename('score')
    
    best_of = data["best_of"]\
        .astype(int)\
        .rename('best_of')
    
    round = data["round"]\
        .astype(str)\
        .rename('round')

    num_sets = data['score']\
        .apply(lambda x: len(x.split()) if isinstance(x, str) else 0)\
        .rename('num_sets')
    

    winner_id = data['winner_id']\
        .astype(int)\
        .rename('winner_id')
    
    winner_name = data['winner_name']\
        .astype(str)\
        .rename('winner_name')
    
    # winner_rank_points = data['winner_rank_points']\
    #     .astype(int)\
    #     .rename('winner_rank_points')
    
    loser_id = data['loser_id']\
        .astype(int)\
        .rename('loser_id')
    
    loser_name = data['loser_name']\
        .astype(str)\
        .rename('loser_name')
    
    # loser_rank_points = data['loser_rank_points']\
    #     .astype(int)\
    #     .rename('loser_rank_points')
    
    cleaned_df = pd.concat(
        [   
            match_id,
            match_year,
            tourney_id, 
            tourney_name, 
            surface, 
            draw_size, 
            tourney_level, 
            tourney_date, 
            match_num, 
            score, 
            best_of, 
            round, 
            num_sets,
            winner_id, 
            winner_name, 
            # winner_rank_points, 
            loser_id, 
            loser_name],
            # loser_rank_points], 
         axis=1)
    
    assert cleaned_df.isna().sum().sum() == 0,\
        f'There are {cleaned_df.isna().sum()} nan values'
    
    return cleaned_df



#### Treating data


In [270]:
dataframe = load_all_df('../data/atp_matches')

100%|██████████| 54/54 [00:04<00:00, 13.32it/s]


In [271]:
clean_data = clean_df(dataframe)

In [272]:
clean_data.columns

Index(['match_id', 'match_year', 'tourney_id', 'tourney_name', 'surface',
       'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'score',
       'best_of', 'round', 'num_sets', 'winner_id', 'winner_name', 'loser_id',
       'loser_name'],
      dtype='object')

In [273]:
clean_data.isna().sum()

match_id         0
match_year       0
tourney_id       0
tourney_name     0
surface          0
draw_size        0
tourney_level    0
tourney_date     0
match_num        0
score            0
best_of          0
round            0
num_sets         0
winner_id        0
winner_name      0
loser_id         0
loser_name       0
dtype: int64

#### Save data to csv

In [274]:
clean_data.to_csv('../modeling/data/clean_atp_matches.csv')
clean_data.to_csv('../web/web_data/clean_atp_matches.csv')

# Players

In [275]:
players = pd.read_csv('../data/atp_players.csv')

/var/folders/s3/s4gmfm1j23xg8_5fnbsqlflm0000gp/T/ipykernel_11434/1952885467.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  players = pd.read_csv('../data/atp_players.csv')


No hi ha cap na a player_id però tenim jugadors sense noms ni cognoms...

In [276]:
players['player_id'].isna().sum()

np.int64(0)

In [277]:
players[['name_first', 'name_last']].isna().all(axis=1).sum()

np.int64(48)

In [278]:
def clean_players(players):

    # Faig drop na de player_id però no n'hi ha cap. 
    player_id = players['player_id']\
        .dropna()\
        .astype(int)\
        .rename('player_id')

    name_first = players['name_first']\
        .fillna('Desconegut')\
        .astype(str)\
        .rename('First Name')

    name_last = players['name_last']\
        .fillna('Desconegudez')\
        .astype(str)\
        .rename('Last Name')

    hand = players['hand']\
        .fillna('U')\
        .astype(str)\
        .rename('Hand')

    country = players['ioc']\
        .fillna('UNK')\
        .astype(str)\
        .rename('Country')

    height = players['height']\
        .fillna(0)\
        .astype(int)\
        .rename('Height')

    clean_players = pd.concat(
        [
        name_first,
        name_last,
        hand,
        country,
        height],
        axis=1
    ).set_index(player_id)

    return clean_players

clean_players = clean_players(players)
clean_players.to_csv('../modeling/data/clean_atp_players.csv')
clean_players.to_csv('../web/web_data/clean_atp_players.csv')

In [279]:
clean_players

,First Name,Last Name,Hand,Country,Height
player_id,,,,,
100001,Gardnar,Mulloy,R,USA,185
100002,Pancho,Segura,R,ECU,168
100003,Frank,Sedgman,R,AUS,180
100004,Giuseppe,Merlo,R,ITA,0
100005,Richard,Gonzalez,R,USA,188
...,...,...,...,...,...
213700,Matvei,Kobiakov,U,RUS,0
213701,Tobia Costanzo,Baragiola Mordini,U,ITA,0
213702,Dominik,Wijntjes,U,NZL,0


In [280]:
clean_players.to_csv('../modeling/data/clean_atp_players.csv')
clean_players.to_csv('../web/web_data/clean_atp_players.csv')

# Tests


In [281]:
web_players = pd.read_csv('../web/web_data/clean_atp_players.csv')

In [282]:
web_players.isna().sum()

player_id     0
First Name    0
Last Name     0
Hand          0
Country       0
Height        0
dtype: int64